In [6]:
import numpy as np
import numpy.lib.recfunctions as recfun
from pathlib import Path
from plyfile import PlyData, PlyElement
import matplotlib.pyplot as plt
import open3d as o3d

from matplotlib import cm

import random
import os

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import MulticlassF1Score, MulticlassPrecision, MulticlassRecall, MulticlassJaccardIndex, BinaryAccuracy, BinaryMatthewsCorrCoef

import torchsummary
import time

In [7]:
def set_seed(seed=42):
    # Python & OS
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch CPU
    torch.manual_seed(seed)
    
    # PyTorch GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        
    # Backend CUDA (CuDNN)
    #torch.backends.cudnn.deterministic = True
    #torch.backends.cudnn.benchmark = True
    
    #print(f"Global seed fixed on {seed} (Deterministic CuDNN activated)")
    print(f"Global seed fixed on {seed}")

In [20]:
set_seed()

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

Global seed fixed on 42


In [ ]:
def load_dataset_ply_lb(dataset='Train', base_dir='../data/Challenge-ABC'):
    base_path = Path(base_dir) / dataset
    if not base_path.exists():
        print(f"Error: Dataset folder {base_path} not found.")
        return None, None, None

    ply_dir = base_path / 'ply'
    lb_dir = base_path / 'lb'

    ply_files = list(ply_dir.glob('*.ply'))
    total_files = len(ply_files)

    points_list = []
    labels_list = []
    file_ids = []

    for i, ply_file in enumerate(ply_files, 1):
        file_id = ply_file.stem
        lb_file = lb_dir / f"{file_id}.lb"

        # Check that the labels file exists
        if not lb_file.exists():
            print(f"Error: Didn't find labels file for {file_id}.ply -> ignored.")
            continue

        # Load labels
        labels = np.loadtxt(lb_file, dtype='int')
        
        # Load point cloud using Open3D
        pcd = o3d.io.read_point_cloud(str(ply_file))
        points = np.asarray(pcd.points)

        # Check dimension consistency
        if len(points) != len(labels):
            print(f"Error: Dimension error with {file_id} : {len(points)} points vs {len(labels)} labels -> ignored.")
            continue

        # Append to lists to maintain object separation
        points_list.append(points)
        labels_list.append(labels)
        file_ids.append(file_id)

        # Progress tracking
        if i % 20 == 0:
            print(f"Loading : {i}/{total_files} files...")
            
    print(f"Successfully loaded {len(file_ids)}/{total_files} files.")

    return points_list, labels_list, file_ids

In [10]:
train_pts, train_lb, train_ids = load_dataset_ply_lb(dataset='Train')

Loading : 20/198 files...
Loading : 40/198 files...
Loading : 60/198 files...
Loading : 80/198 files...
Loading : 100/198 files...
Loading : 120/198 files...
Loading : 140/198 files...
Loading : 160/198 files...
Loading : 180/198 files...
Successfully loaded 198/198 files.


In [ ]:
def downsample_non_edges(points, labels, method, display=False, **kwargs):
    """
    Downsamples the non-edge points (label 0) of a point cloud while preserving all edge points (label 1).
    
    Parameters:
    - points: (N, 3) numpy array of spatial coordinates.
    - labels: (N,) numpy array of binary labels (1 for edge, 0 for non-edge).
    - method: String specifying the down-sampling algorithm ('voxel', 'fps', 'poisson', 'random', 'uniform').
    - display: Boolean. If True, renders the intermediate and final point clouds via plotly.
    - **kwargs: Method-specific parameters:
        - voxel: 'resolution_percentage' (e.g., 0.02 for 2% of max dimension)
        - fps: 'num_points' (int)
        - poisson: 'radius' (float)
        - random: 'num_points' (int)
        - uniform: 'k_step' (int)
    
    Returns:
    - final_global_idx: (M,) numpy array of the globally sorted indices to keep.
    """
    # 1. Initialize structure and extract global masks
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    global_edge_idx = np.where(labels == 1)[0]
    global_non_edge_idx = np.where(labels != 1)[0] # Security; using != 1 to catch any unclassified points
    
    non_edge_pcd = pcd.select_by_index(global_non_edge_idx)
    
    # 2. Routing logic for the selected algorithm
    method = method.lower()
    if method == 'voxel':
        if 'resolution_percentage' not in kwargs:
            raise ValueError("Method 'voxel' requires 'resolution_percentage' (e.g., 0.02).")
        
        # Calculate bounds on the full point cloud to maintain consistent relative scale
        min_bound = points.min(axis=0)
        max_bound = points.max(axis=0)
        max_dim = np.max(max_bound - min_bound)
        dynamic_voxel_size = max_dim * kwargs['resolution_percentage']
        
        _, _, trace = non_edge_pcd.voxel_down_sample_and_trace(
            voxel_size=dynamic_voxel_size, 
            min_bound=min_bound, 
            max_bound=max_bound
        )
        rel_idx = np.array([v[np.random.randint(0, len(v))] for v in trace])
        selected_non_edge_global_idx = global_non_edge_idx[np.sort(rel_idx)]
        
    elif method == 'fps':
        if 'num_points' not in kwargs:
            raise ValueError("Method 'fps' requires 'num_points' keyword argument.")
        
        fps_pcd = non_edge_pcd.farthest_point_down_sample(kwargs['num_points'])
        tree = o3d.geometry.KDTreeFlann(non_edge_pcd)
        rel_idx = np.zeros(kwargs['num_points'], dtype=int)
        
        for i, pt in enumerate(fps_pcd.points):
            rel_idx[i] = tree.search_knn_vector_3d(pt, 1)[1][0]
            
        selected_non_edge_global_idx = global_non_edge_idx[rel_idx]

    elif method == 'poisson':
        if 'radius' not in kwargs:
            raise ValueError("Method 'poisson' requires 'radius' keyword argument.")
        
        tree = o3d.geometry.KDTreeFlann(non_edge_pcd)
        pts = np.asarray(non_edge_pcd.points)
        num_pts = len(pts)
        
        active_mask = np.ones(num_pts, dtype=bool)
        shuffled_idx = np.random.permutation(num_pts)
        rel_idx = []
        
        for idx in shuffled_idx:
            if not active_mask[idx]: continue
            
            rel_idx.append(idx)
            _, neighbors_idx, _ = tree.search_radius_vector_3d(pts[idx], kwargs['radius'])
            active_mask[np.asarray(neighbors_idx)] = False
            
        selected_non_edge_global_idx = global_non_edge_idx[rel_idx]

    elif method == 'random':
        if 'num_points' not in kwargs:
            raise ValueError("Method 'random' requires 'num_points' keyword argument.")
        
        target_pts = min(kwargs['num_points'], len(global_non_edge_idx))
        selected_non_edge_global_idx = np.random.choice(global_non_edge_idx, size=target_pts, replace=False)
        selected_non_edge_global_idx = np.sort(selected_non_edge_global_idx)

    elif method == 'uniform':
        if 'k_step' not in kwargs:
            raise ValueError("Method 'uniform' requires 'k_step' keyword argument.")
            
        selected_non_edge_global_idx = global_non_edge_idx[::kwargs['k_step']]

    else:
        raise ValueError(f"Unknown method '{method}'. Valid options: voxel, fps, poisson, random, uniform.")

    # 3. Final Reintegration
    final_global_idx = np.concatenate([global_edge_idx, selected_non_edge_global_idx])
    final_global_idx = np.sort(final_global_idx)

    # 4. Diagnostics and Visualization
    if display:
        print(f"\n--- Downsampling Report ({method.upper()}) ---")
        print(f"Original Cloud Size : {len(points)}")
        print(f"Edge Points Kept    : {len(global_edge_idx)}")
        print(f"Non-Edge Downsampled: {len(selected_non_edge_global_idx)}")
        print(f"Final Cloud Size    : {len(final_global_idx)}")
        
        if method == 'voxel':
            print(f"Computed Voxel Size : {dynamic_voxel_size:.4f}")
        
        inter_pcd = pcd.select_by_index(selected_non_edge_global_idx)
        res_pcd = pcd.select_by_index(final_global_idx)
        
        res_pcd.paint_uniform_color([0, 0, 1])
        inter_pcd.paint_uniform_color([0, 0, 1])
        edge_colors = np.zeros((len(final_global_idx), 3))
        edge_colors[:] = [0, 0, 1]
        
        edge_mask_in_final = np.isin(final_global_idx, global_edge_idx)
        edge_colors[edge_mask_in_final] = [1, 0, 0] 
        res_pcd.colors = o3d.utility.Vector3dVector(edge_colors)
        

        o3d.visualization.draw_plotly([inter_pcd])
        o3d.visualization.draw_plotly([res_pcd])

    return final_global_idx

In [ ]:
def build_downsampled_features(points_list, labels_list, file_ids, method, dataset='Train', base_dir='../data/Challenge-ABC', **kwargs):
    ssm_dir = Path(base_dir) / dataset / 'SSM_Challenge-ABC'
    
    X_list = []
    y_list = []
    
    total_files = len(file_ids)
    print(f"Starting feature extraction using '{method}' downsampling...")

    for i in range(total_files):
        file_id = file_ids[i]
        points = points_list[i]
        labels = labels_list[i]
        
        # Compute the global indices to keep for this specific model
        final_idx = downsample_non_edges(points, labels, method=method, display=False, **kwargs)
        
        # Load the corresponding .ssm file
        ssm_file = ssm_dir / f"{file_id}.ssm"
        if not ssm_file.exists():
            print(f"Error: {ssm_file.name} not found -> ignored.")
            continue
        features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
        
        # Dimension consistency check
        if len(features) != len(points):
            print(f"Error: Dimension mismatch in {file_id}.ssm ({len(features)} features vs {len(points)} points) -> ignored.")
            continue
            
        # Slice the arrays using the downsampled indices
        filtered_features = features[final_idx]
        filtered_labels = labels[final_idx]
        
        X_list.append(filtered_features)
        y_list.append(filtered_labels)
        
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{total_files} feature files...")
            
    print(f"Feature extraction complete. Processed {len(X_list)} valid files.")
    
    return X_list, y_list

In [24]:
train_pts, train_lb, train_ids = load_dataset_ply_lb(dataset='Train')

X_train_list, y_train_list = build_downsampled_features(
    points_list=train_pts,
    labels_list=train_lb,
    file_ids=train_ids,
    method='voxel',
    resolution_percentage=0.02
)

X_train_flat = np.vstack(X_train_list)
y_train_flat = np.concatenate(y_train_list)

print(f"Final training tensors shape: \nX_train{X_train_flat.shape} \ny_train{y_train_flat.shape} ")

Loading : 20/198 files...
Loading : 40/198 files...
Loading : 60/198 files...
Loading : 80/198 files...
Loading : 100/198 files...
Loading : 120/198 files...
Loading : 140/198 files...
Loading : 160/198 files...
Loading : 180/198 files...
Successfully loaded 198/198 files.
Starting feature extraction using 'voxel' downsampling...
Processed 10/198 feature files...
Processed 20/198 feature files...
Processed 30/198 feature files...
Processed 40/198 feature files...
Processed 50/198 feature files...
Processed 60/198 feature files...
Processed 70/198 feature files...
Processed 80/198 feature files...
Processed 90/198 feature files...
Processed 100/198 feature files...
Processed 110/198 feature files...
Processed 120/198 feature files...
Processed 130/198 feature files...
Processed 140/198 feature files...
Processed 150/198 feature files...
Processed 160/198 feature files...
Processed 170/198 feature files...
Processed 180/198 feature files...
Processed 190/198 feature files...
Feature extr

In [22]:
def load_full_dataset(dataset='Train'):
    # 1) Path Definition
    # base directory for the chosen dataset
    base_path = Path('../data/Challenge-ABC') / dataset
    if not base_path.exists(): # verification
        print(f"Error: Dataset folder {base_path} not found, check 'dataset' argument.")
        return None
    ssm_dir = base_path / 'SSM_Challenge-ABC'
    lb_dir = base_path / 'lb'

    X_list = []
    y_list = []

    ssm_files = list(ssm_dir.glob('*.ssm'))
    total_files = len(ssm_files)

    for i, ssm_file in enumerate(ssm_files, 1):
        file_id = ssm_file.stem
        lb_file = lb_dir / f"{file_id}.lb"

        # check that the labels file exists
        if not lb_file.exists():
            print(f"Error: Didn't find labels file for {file_id}.ply -> ignored.")
            continue

        features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
        labels = np.loadtxt(lb_file, dtype='int')

        if len(labels) == len(features):
            X_list.append(features)
            y_list.append(labels)
        else:
            print(f"Error: Dimension error for {file_id} -> ignored.")

        if i % 20 == 0:
            print(f"Loading : {i}/{total_files} files...")

    X = np.vstack(X_list)
    y = np.concatenate(y_list)    
    
    return X, y

In [23]:
X_val, y_val = load_full_dataset('Validation')

Loading : 20/50 files...
Loading : 40/50 files...
